In [1]:
from mesh.core import *

from hyperopt import fmin, tpe, hp
from pymoo.indicators.hv import HV
from pymoo.problems import get_problem

## Create the hyperparameter optimizer for MESH:

In [ ]:
# Define the objective/fitness function
experiment_name = 'dtlz1'

# MESH configuration that won't be optimized
objective_dim = 2 # Number of objectives
position_dim = 10 # Design space dimension
position_min_value = np.array([0]*position_dim) # Lower bound of problem
position_max_value = np.array([1]*position_dim) # Upper bound of problem

max_iterations = 0 # Maximum number of iterations (not used if it less than one)
max_fitness_eval = 15000 # Maximum fitness evaluations (not used if it is less than one)
population_size = 50 # Population size
num_final_solutions = population_size # Number of final solutions
memory_size = population_size # Maximum number of particles in memory

global_best_attribution_type = 0 # 0 -> E1 | 1 -> E2 | 2 -> E3 | 3 -> E4
dm_pool_type = 0 # 0 -> V1 | 1 -> V2 | 2 -> V3
dm_operation_type = 0 # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)

random_state = None # Defines a seed for random numbers (not used if it is None)

# Function used by hyperopt to optimize MESH for a function by the hypervolume
def optimize_hyperparameter_by_hypervolume(function, hyperparameters, indicator):
    # Get the hyperparameters from the optimizer
    communication_probability = hyperparameters["communication_probability"]
    mutation_rate = hyperparameters["mutation_rate"]
    personal_guide_array_size = hyperparameters["personal_guide_array_size"]
    # Configure the MESH to the hyperparameters
    mesh_params = MeshParameters(objective_dim,
                                 position_dim, position_min_value, position_max_value, 
                                 population_size, memory_size=memory_size,
                                 global_best_attribution_type=global_best_attribution_type,
                                 dm_pool_type=dm_pool_type,
                                 dm_operation_type=dm_operation_type,
                                 communication_probability=communication_probability, mutation_rate=mutation_rate,
                                 max_gen=max_iterations, max_fit_eval=max_fitness_eval,
                                 max_personal_guides=personal_guide_array_size,
                                 random_state=random_state)
    mesh = Mesh(mesh_params, function)
    # Run MESH
    mesh.run()
    # Get the results from MESH
    Pos, Fit = mesh.get_results()
    # Calculate the hypervolume
    hypervolume = indicator(Fit)
    return -hypervolume

## Apply the fine tuning to the MESH:

In [7]:
# Definition of design space of hyperparameters
hyperperameter_space = {
    "communication_probability": hp.uniform("communication_probability", 0, 1),
    "mutation_rate": hp.uniform("mutation_rate", 0, 1),
    "personal_guide_array_size": hp.choice("personal_guide_array_size", [1, 2, 3])
}

# Define the objective/fitness function
if(experiment_name in {'zdt1', 'zdt2', 'zdt3', 'zdt4', 'zdt6'}):
	func = get_problem(experiment_name, n_var=position_dim).evaluate
else:
	func = get_problem(experiment_name, n_var=position_dim, n_obj=2).evaluate

# Define the indicator to calculate the volume
ref_point = np.array([11, 11])
indicator = HV(ref_point=ref_point)

best = fmin(lambda params: optimize_hyperparameter_by_hypervolume(func, params, indicator), hyperperameter_space, algo=tpe.suggest, max_evals=100, verbose=0)
print("Best hyperparameters:", best)

Best hyperparameters: {'communication_probability': np.float64(0.2472496296707432), 'mutation_rate': np.float64(0.5588679245954559), 'personal_guide_array_size': np.int64(0)}
